# V5 SE-OI: piloto CPU reproducible

**Candidata fijada: `seoi_mse`.** Escala causal de seis retornos conocidos y reconstrucción dentro de la ley de volatilidad solapada. Seis recetas por cinco semillas: **30ajustes nuevos**, sin HPO. Este notebook se entrega sin entrenamientos ejecutados.

El piloto compara una modificación concreta; no promete superioridad anticipada. Los datos son desarrollo previamente expuesto. Los796trabajos aditivos y los pilotos anteriores se conservan.

Usa un runtime CPU. La instalación se hace en un entorno separado del Torch de Colab; no se requiere A100. No abras dos runtimes sobre la misma carpeta de Drive.

Las fuentes, datos y regla están fijados por commit y SHA256. La preparación ejecuta36pruebas sintéticas, verifica datos y registra el caso. Sólo la celda3entrena.


## 1. Conserva el caso en Drive
Autoriza el montaje de tu Drive en Colab para que una desconexión no elimine checkpoints.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Prepara y verifica; cero entrenamientos
La primera instalación puede tardar varios minutos y descarga aproximadamente200–350MB. La carpeta es nueva V5; no uses el master de796trabajos ni los casos V3/V4.


In [ ]:
import hashlib
import urllib.request
from pathlib import Path

SOURCE_COMMIT = '8777477fcde6c34890e447cc801df2f9c60a43c1'
SOURCE_MANIFEST_SHA256 = '7f32590bb954fb110a8755525c3ba74ef50fb4ef15e116f0274bad7a957a3dc8'
BOOTSTRAP_SHA256 = 'd687e6c0075af80230972ed930b67ec60edd4e68428a45a8d3a7ff16ae7d95c4'
CASE_ROOT = "/content/drive/MyDrive/DeepLearning_E3/SE_OI_V5/pilot_01"

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Ejecuta primero la celda que monta Drive.")
url = (
    "https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/"
    + SOURCE_COMMIT
    + "/experiments/e3_seoi_v5_pilot_20261121/bootstrap_colab.py"
)
with urllib.request.urlopen(url, timeout=60) as response:
    code = response.read(2 * 1024 * 1024)
if hashlib.sha256(code).hexdigest() != BOOTSTRAP_SHA256:
    raise RuntimeError("El código no coincide con la versión fijada; no se ejecuta.")
E3_MODULE = {"__name__": "e3_v5_pinned_setup"}
exec(compile(code, "e3_v5_pinned_setup", "exec"), E3_MODULE)
E3 = E3_MODULE["setup"](
    source_commit=SOURCE_COMMIT,
    manifest_sha256=SOURCE_MANIFEST_SHA256,
    case_root=CASE_ROOT,
)


## 3. Ejecuta o reanuda el piloto
Hasta40épocas, paciencia8 y120s activos por ajuste. Se conservan fallos, checkpoints por época y contabilidad de interrupciones. El límite cooperativo puede excederse en una operación; las escrituras a Drive se miden aparte.


In [ ]:
# Esta celda sí inicia o reanuda los30ajustes nuevos de V5.
# No ejecuta la evaluación de desarrollo ni cambia de candidata.
E3_MODULE["command"](E3, "run")


## 4. Evalúa una sola versión
Se exigen los30trabajos completos. Regla fija: mejora≥2% frente al mejor control,≥4/5victorias frente al mejor control neuronal por RMSE y MAE≤1,05veces la de ese mismo control. No hay promoción automática de una ablación.


In [ ]:
# Sólo funciona cuando los30ajustes están completos y verificados.
E3_MODULE["command"](E3, "assess")


## 5. Exporta tablas y recibos
El ZIP no incluye NPZ, pesos, Adam o RNG. Esta celda también permite conservar diagnósticos si el piloto quedó incompleto. El archivo seguirá en Drive además de la descarga.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import shutil
from google.colab import files

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
public_dir = Path(E3["case_root"]).parent / ("v5-public-" + stamp)
E3_MODULE["command"](E3, "export", "--output", str(public_dir))
archive = shutil.make_archive(str(public_dir), "zip", root_dir=public_dir)
print("Resultados públicos:", archive)
print("SHA256:", hashlib.sha256(Path(archive).read_bytes()).hexdigest())
files.download(archive)


## Estado y recuperación
Para reanudar tras desconexión, repite1y2con la misma carpeta y luego3. Un cambio de código, datos o cohorte no se mezcla con este caso. Si alguna operación falla, consulta el estado y exporta los diagnósticos.


In [ ]:
# Consulta y diagnóstico: no entrena ni vuelve a predecir.
E3_MODULE["command"](E3, "status")
# Si necesitas compartir un fallo, la celda de exportación también funciona
# con un piloto incompleto y excluye los pesos y los datos brutos.


## Alcance del resultado

La preparación de este notebook ejecutó pruebas del software, sin entrenar ni generar nuevos pronósticos de mercado. Los resultados aparecerán sólo después de ejecutar las celdas3y4.

Un piloto favorable permite decidir sobre continuidad: misma candidata, revisión de contribución, familia completa, procedimientos adecuados, dimensionamiento de potencia y datos confirmatorios apropiados. La guía exige evidencia real de superioridad para R3. El piloto usa55fechas de desarrollo ya examinadas; no se convierten en un test independiente.

Asistencia de IA declarada en revisión, formulación, programación y pruebas. No se generaron firmas, una defensa observada o acreditación docente.
